# Vision B's ISLES'24 false alarms: the pre-registered selection on RSNA validation (Item 2, EXP-49, Kaggle)

At DEC-28's threshold Vision B flags 68 of 129 presumed bleed-free ISLES'24 scans; EXP-35 traced half the borderline ones to skull-base slabs. `docs/ITEM2_FALSE_ALARMS.md` (approved by Parsa, 19 Nov 2026, before any score) fixed 10 candidate scan-score rules (`slab_rules.py`: none, gate at four ratios, adjacent, both at four ratios) and how one is chosen. This notebook runs that choice on RSNA validation:

1. EXP-6's `best.pt` on every validation slice, with the scan AUC checked against EXP-6 (as in notebook 22).
2. Each slice's brain fraction (`explain.tissue_masks`), on the CPU workers.
3. `false_alarms.select`: every candidate cross-fitted at DEC-28's 96% target over EXP-17's 20 seeds, sensitivity per bleed subtype, the eligibility rule, the proposal and its paired specificity gain over "none".
4. Grad-CAM on 20 true-positive and 20 false-positive scans' hottest slices at DEC-28's threshold: where the heat falls, as medians.

No model is trained. The logic is in `src/strokecopilot/vision/haemorrhage/false_alarms.py`, `slab_rules.py` and `explain.py`.

**Data rules (`docs/DATA_USE.md`).** RSNA data stays on Kaggle. The split and the reference slices hold IDs, so they are rebuilt here and never saved. Output files hold counts, rates and thresholds only; section 9 checks for IDs before saving. **The RSNA test split stays closed:** nothing here reads it, and `ALLOW_RSNA_TEST` must not be set. Model weights stay private to the team.

**Setup** (as for notebook 22)
1. Upload this notebook to Kaggle (File → Import Notebook).
2. Add Input: the four `rsna-slim-v1-file-s0` to `-s3` datasets, and the private weights dataset holding EXP-6's `best.pt`, `calibration.json` and `metrics.json`.
3. Settings: accelerator **GPU T4 x2**, **Internet on**. Only one GPU is used.

**Run it:** Save Version → **Save & Run All**. EXP-17 took 6.4 min for the copy and predictions; the brain-fraction pass over about 75,500 slices is unmeasured (estimate 5–20 min). Free.

**Done when** section 9 prints `OK:`. Then download `config.json` and `metrics.json` from the version's Output and upload them to `StrokeCopilot/runs/EXP-49_visionB_false_alarms/` on Drive.

## 1. Settings

In [ ]:
EXPERIMENT = 'EXP-49'
DATA_ROOT = '/kaggle/input'
LOCAL_DIR = '/kaggle/temp'    # never /kaggle/working: Save Version would keep the shards as output
OUT_ROOT = '/kaggle/working'
EXPECTED = {'readable_slices': 752802, 'scans': 21744, 'patients': 18938, 'val_scans': 2166}
EXP6_SCAN_AUC = 0.977
BATCH_SIZE = 128
REPO_REF = 'main'
RUN_DIR = f'{OUT_ROOT}/{EXPERIMENT}_visionB_false_alarms'

## 2. Install the package
Installs the package at the current commit of `REPO_REF`.

In [ ]:
import json, os, shutil, subprocess, sys, time

started_all = time.perf_counter()
if os.environ.get('ALLOW_RSNA_TEST') == '1':
    raise RuntimeError('ALLOW_RSNA_TEST is set. The test split stays closed for this task; unset it.')
repo = 'https://github.com/parsa-hke/strokecopilot.git'
found = subprocess.run(['git', 'ls-remote', repo, REPO_REF], capture_output=True, text=True)
if found.returncode or not found.stdout.strip():
    raise RuntimeError(('Could not read the repo or REPO_REF. Check the branch name. ' + found.stderr)[-800:])
COMMIT = found.stdout.split()[0]
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'git+{repo}@{COMMIT}'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
del repo, found, result
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
    import timm

import numpy as np
import torch
from strokecopilot.data import rsna, rsna_split
from strokecopilot.vision.haemorrhage import aggregate as agg
from strokecopilot.vision.haemorrhage import operating_point as op
from strokecopilot.vision.haemorrhage import train
from strokecopilot.vision.haemorrhage import explain as ex, false_alarms as fa, infer, slab_rules as sr

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(f"Package at {COMMIT[:7]}; torch {torch.__version__}; timm {timm.__version__}; GPU {gpu or 'none'}.")
if DEVICE.type != 'cuda':
    raise RuntimeError('No GPU. Settings → Accelerator → GPU T4 x2, then Save & Run All again.')

## 3. EXP-6 model and the run record
Finds `best.pt`, `calibration.json` and EXP-6's `metrics.json` in the attached weights dataset, then writes `config.json` (with the pre-registered candidates and rules) before anything is computed.

In [ ]:
import glob

def find(name):
    hits = sorted({h for depth in range(1, 5) for h in glob.glob(os.path.join(DATA_ROOT, *['*'] * depth, name))})
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected one {name} in the attached datasets, found {len(hits)}. Attach the weights dataset (and only one).')
    return hits[0]

BEST, CALIBRATION, EXP6_METRICS = find('best.pt'), find('calibration.json'), find('metrics.json')
net, state = train.load_best(BEST, DEVICE)
calibrator = agg.Calibrator.from_json(CALIBRATION)
op.check_monotonic(calibrator)
exp6 = json.load(open(EXP6_METRICS))
if exp6.get('experiment') != 'EXP-6':
    raise ValueError(f"The attached metrics.json is from {exp6.get('experiment')}, not EXP-6.")
EXP6_FINGERPRINT = exp6['split']['fingerprint']
print(f"Model {state['config']['model']} from epoch {state['epoch']}; aggregation {calibrator.method} k={calibrator.k}, "
      f"{calibrator.kind} calibration, threshold {calibrator.threshold:.3f}. EXP-6 split fingerprint {EXP6_FINGERPRINT[:12]}.")

if os.path.exists(os.path.join(RUN_DIR, 'metrics.json')):
    raise FileExistsError(f'{RUN_DIR} already has metrics.json. Use a new Experiments row.')
os.makedirs(RUN_DIR, exist_ok=True)
config = {'experiment': EXPERIMENT, 'task': 'Item 2: ISLES false alarms, pre-registered selection on RSNA validation', 'commit': COMMIT,
          'repo_ref': REPO_REF, 'preregistration': 'docs/ITEM2_FALSE_ALARMS.md (approved by Parsa, 19 Nov 2026)',
          'model_from': 'EXP-6', 'epoch': state['epoch'], 'calibration': vars(calibrator), 'split_seed': rsna_split.SEED,
          'split_fingerprint': EXP6_FINGERPRINT, 'half_seed': op.SEED, 'repeat_seeds': [op.REPEAT_SEEDS[0], op.REPEAT_SEEDS[-1]],
          'target_sensitivity': fa.TARGET, 'candidates': [fa.label(r, q) for r, q in sr.candidates()], 'gate_ratios': list(sr.GATE_RATIOS),
          'k': sr.K, 'sensitivity_margin': fa.SENS_MARGIN, 'subtype_margin': fa.SUBTYPE_MARGIN, 'subtypes': list(fa.SUBTYPES),
          'reference_per_group': fa.REFERENCE_PER_GROUP, 'reference_threshold': infer.CONTRACT_THRESHOLD,
          'masks': {'air_below_hu': ex.AIR_BELOW_HU, 'bone_above_hu': ex.BONE_ABOVE_HU, 'edge_px': ex.EDGE_PX, 'bone_margin_px': ex.BONE_MARGIN_PX},
          'bootstrap': op.BOOTSTRAP, 'test_split': 'closed', 'gpu': gpu}
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=1, default=str)
print(f'config.json written to {RUN_DIR}.')

## 4. RSNA shards
The same steps as notebooks 20 and 22: shards are copied to the local disk and checked against their index.

In [ ]:
started = time.perf_counter()
shards = rsna.find_shards(DATA_ROOT)
index = rsna.load_index(shards)
readers = {}
for shard, (path, _) in shards.items():
    if os.path.isdir(path):
        raise ValueError(f'Shard {shard} is attached as a folder. Attach the rsna-slim-v1-file-s{shard} dataset instead.')
    local_path, _ = rsna.copy_shard(path, LOCAL_DIR)
    readers[shard] = rsna.ShardReader(local_path)
    if readers[shard].ids() != set(index.loc[index['shard'] == shard, 'image_id']):
        raise ValueError(f'Shard {shard}: its images and index differ. Check that dataset.')
SETUP_SECONDS = round(time.perf_counter() - started)
print(f'{len(index):,} readable slices in {len(readers)} shards, set up in {SETUP_SECONDS} s.')
if len(index) != EXPECTED['readable_slices']:
    raise ValueError(f"{len(index):,} readable slices, expected {EXPECTED['readable_slices']:,}.")

## 5. The validation split, exactly as EXP-6 built it
The split is rebuilt with its fixed seed and must match EXP-6's fingerprint. Only the validation rows are kept.

In [ ]:
scans = rsna.scan_table(index)
assignment = rsna_split.assign(scans)
FINGERPRINT = rsna_split.fingerprint(assignment)
if FINGERPRINT != EXP6_FINGERPRINT:
    raise ValueError('The rebuilt split differs from EXP-6 (fingerprints differ). Stop and tell Parsa.')
scans = rsna_split.add_split(scans, assignment)
index = index.merge(scans[['series_id', 'split']], on='series_id', validate='many_to_one')
val_items = rsna_split.rows(index, 'val').reset_index(drop=True)
known_ids = set(scans['patient_id']) | set(scans['series_id'])
del assignment, scans, index
n_val_scans = val_items['series_id'].nunique()
print(f"Validation: {val_items['patient_id'].nunique():,} patients, {n_val_scans:,} scans, {len(val_items):,} slices. Split matches EXP-6.")
if n_val_scans != EXPECTED['val_scans']:
    raise ValueError(f"{n_val_scans} validation scans, expected {EXPECTED['val_scans']}.")

## 6. Validation slice predictions
EXP-6's `best.pt` on every validation slice, then a check that the scan AUC matches EXP-6. If it does not, nothing after this is trustworthy.

In [ ]:
WORKERS = max(1, os.cpu_count() or 2)
started = time.perf_counter()
val_items = val_items.assign(prob=train.predict(net, train.SliceDataset(val_items, readers), DEVICE, BATCH_SIZE, WORKERS))
PREDICT_SECONDS = round(time.perf_counter() - started)
check = agg.evaluate(val_items, calibrator)
print(f"{check['slices']:,} slices scored in {PREDICT_SECONDS} s. Slice AUC {check['slice_auc']:.4f}, scan AUC {check['scan_auc']:.4f} "
      f"(EXP-6 {EXP6_SCAN_AUC}); at {calibrator.threshold:.3f}: sensitivity {check['sensitivity']:.3f}, specificity {check['specificity']:.3f} "
      f"(EXP-6 0.951 and 0.917, same split).")
if abs(check['scan_auc'] - EXP6_SCAN_AUC) > 0.005:
    raise ValueError(f"Scan AUC {check['scan_auc']:.4f} is not EXP-6's {EXP6_SCAN_AUC}. Check the weights dataset, then tell Parsa.")

## 7. Brain fraction per slice
`explain.tissue_masks` on every validation slice, on the CPU workers: brain-density pixels as a share of the head's area. Only its distribution is printed.

In [ ]:
started = time.perf_counter()
measured = fa.measure(val_items, readers, workers=WORKERS)
val_items = val_items.join(measured)
val_items['height'] = fa.heights(val_items)
MEASURE_SECONDS = round(time.perf_counter() - started)
bf = val_items['brain_fraction']
MEASURE_SUMMARY = {'slices': int(len(bf)), 'no_head': int((val_items['head_px'] == 0).sum()), 'z_missing': int(val_items['z_mm'].isna().sum()),
                   'brain_fraction_percentiles': {p: round(float(bf.quantile(p / 100)), 4) for p in (5, 25, 50, 75, 95)}}
print(f"Measured {len(bf):,} slices in {MEASURE_SECONDS} s. {MEASURE_SUMMARY}")

## 8. The pre-registered selection
Every candidate, cross-fitted at 96% over 20 seeds; the eligibility rule (sensitivity no more than 1 point below "none", no subtype more than 3 points below); the proposal; and its paired specificity gain over "none". Then Grad-CAM on the reference slices.

In [ ]:
started = time.perf_counter()
cols = ['series_id', 'patient_id', 'any', *fa.SUBTYPES, 'z_mm', 'prob', 'brain_fraction']
report = fa.select(val_items[cols])
SELECT_SECONDS = round(time.perf_counter() - started)
print(f"\n{'Candidate':12s} | sensitivity | specificity | subtype sensitivity (epi, ipa, ivh, sah, sdh) | eligible")
for name, r in report['candidates'].items():
    sub = ', '.join('n/a' if r['subtype_sensitivity'][s] is None else f"{r['subtype_sensitivity'][s]:.3f}" for s in fa.SUBTYPES)
    print(f"{name:12s} | {r['sensitivity']:.4f} | {r['specificity']:.4f} | {sub} | {'yes' if r['eligible'] else 'no: ' + '; '.join(r['ineligible_because'])}")
g = report['specificity_gain']
print(f"\nProposal: {report['proposal']}" + ('' if g is None else f"; specificity gain over none {g['gain']:+.4f} [{g['ci95'][0]:+.4f}, {g['ci95'][1]:+.4f}]")
      + f". Meets the adoption rule: {report['meets_adoption_rule']} (adoption also needs Parsa's decision).")

started = time.perf_counter()
contract = infer.contract_calibrator(CALIBRATION)
ref = fa.reference_slices(val_items, contract, contract.threshold)
images = np.stack([readers[int(s)].read(i) for s, i in zip(ref['shard'], ref['image_id'])])
explained, _ = ex.explain_slices(net, images, DEVICE)
REFERENCE = fa.reference_summary(ref, explained)
GRADCAM_SECONDS = round(time.perf_counter() - started)
del ref, images
print('\nGrad-CAM reference (medians):', json.dumps(REFERENCE, indent=1))

## 9. Save and check
`metrics.json` goes in the output folder with `config.json`. Nothing is saved if an ID reaches a file.

In [ ]:
metrics = {'experiment': EXPERIMENT, 'commit': COMMIT, 'split_fingerprint': FINGERPRINT, 'prediction_check': check,
           'measures': MEASURE_SUMMARY, 'selection': report, 'gradcam_reference': REFERENCE,
           'seconds': {'setup': SETUP_SECONDS, 'predict': PREDICT_SECONDS, 'measure': MEASURE_SECONDS, 'select': SELECT_SECONDS,
                       'gradcam': GRADCAM_SECONDS}, 'minutes_total': round((time.perf_counter() - started_all) / 60, 1)}
texts = {'metrics.json': json.dumps(metrics, indent=1, default=str), 'config.json': open(os.path.join(RUN_DIR, 'config.json')).read()}
known_ids |= set(val_items['image_id'])
for name, text in texts.items():
    if 'ID_' in text or any(i in text for i in known_ids):
        raise ValueError(f'An ID reached {name}. Nothing was saved; stop and check.')
for name, text in texts.items():
    with open(os.path.join(RUN_DIR, name), 'w') as f:
        f.write(text)
for reader in readers.values():
    reader.close()
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
del val_items, known_ids
print('Saved:', sorted(os.listdir(RUN_DIR)))
none, best = report['candidates']['none'], report['candidates'][report['proposal']]
print(f"OK: {EXPERIMENT} at {COMMIT[:7]}: {report['scans']:,} validation scans; none: sensitivity {none['sensitivity']:.4f}, specificity "
      f"{none['specificity']:.4f}; proposal {report['proposal']}: sensitivity {best['sensitivity']:.4f}, specificity {best['specificity']:.4f}; "
      f"meets adoption rule {report['meets_adoption_rule']}; {metrics['minutes_total']} min. Test split not opened.")